In [ ]:
from pathlib import Path
import numpy as np
import polars as pl
from sklearn.ensemble import RandomForestRegressor
projectPath = Path.cwd().resolve()
while projectPath.parent != projectPath and not (projectPath / 'Data').exists():
    projectPath = projectPath.parent
dataPrepPath = projectPath / 'pipelines' / 'dataprep' / 'output'
outputPath = projectPath / 'pipelines' / 'optional' / 'output'
outputPath.mkdir(parents=True, exist_ok=True)
evaluationRows = pl.read_parquet(dataPrepPath / 'evaluationRows.parquet').filter(pl.col('role') == 'pool')
featureRows = pl.read_parquet(dataPrepPath / 'featureSets.parquet')
splitAssignments = pl.read_parquet(dataPrepPath / 'reviewerSplitAssignments.parquet')
featureNames = featureRows.filter(pl.col('featureSet') == 'F0+F1').sort('featureOrder').get_column('featureName').to_list()
makeModel = lambda seed: RandomForestRegressor(n_estimators=300, max_depth=16, min_samples_leaf=5, max_features='sqrt', random_state=seed, n_jobs=-1)
predictionRecords = []
for splitKey, assignmentRows in splitAssignments.partition_by(['splitScheme', 'splitSeed'], as_dict=True).items():
    splitScheme, splitSeed = splitKey
    experimentRows = evaluationRows.join(assignmentRows.select('rowId', 'fold'), on='rowId', how='inner')
    for fold in sorted(experimentRows.get_column('fold').unique().to_list()):
        trainingRows = experimentRows.filter(pl.col('fold') != fold)
        testRows = experimentRows.filter(pl.col('fold') == fold)
        model = makeModel(int(splitSeed))
        model.fit(trainingRows.select(featureNames).to_numpy(), trainingRows.get_column('ndvi').to_numpy())
        predictedValues = model.predict(testRows.select(featureNames).to_numpy())
        predictionRecords.extend({'rowId': int(rowId), 'splitScheme': splitScheme, 'splitSeed': int(splitSeed), 'fold': int(fold), 'prediction': float(prediction), 'ndvi': float(ndvi)} for rowId, prediction, ndvi in zip(testRows.get_column('rowId'), predictedValues, testRows.get_column('ndvi')))
splitPredictions = pl.DataFrame(predictionRecords).with_columns((pl.col('prediction') - pl.col('ndvi')).alias('error')).sort('splitScheme', 'splitSeed', 'fold', 'rowId')
splitStrategyComparison = splitPredictions.group_by('splitScheme', 'splitSeed').agg(pl.col('error').abs().mean().alias('mae'), pl.col('error').pow(2).mean().sqrt().alias('rmse'), (1 - pl.col('error').pow(2).mean() / pl.col('ndvi').var(ddof=0)).alias('r2'), pl.col('error').mean().alias('bias'), pl.len().alias('rowCount'), pl.col('fold').n_unique().alias('foldCount')).sort('splitScheme', 'splitSeed')
splitPredictions.write_parquet(outputPath / 'splitPredictions.parquet')
splitStrategyComparison.write_parquet(outputPath / 'splitStrategyComparison.parquet')
assert splitStrategyComparison.filter(pl.col('splitScheme') == 'random').height == 5
assert splitStrategyComparison.filter(pl.col('splitScheme') == 'groupedWindow').height == 5
assert splitStrategyComparison.filter(pl.col('splitScheme') == 'spatialTile').height == 1
assert splitStrategyComparison.filter(pl.col('rowCount') != evaluationRows.height).is_empty()
splitStrategyComparison
